# MSME-RL — GRPO training on Colab

**OpenEnv hackathon submission** | One-click re-run for judges.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/di35117/msme_env/blob/main/train_colab.ipynb)

| Component | Detail |
|-----------|--------|
| Environment | In-process [`MSMERLEnvironment`](https://github.com/di35117/msme_env) (same logic as the FastAPI Space server; Colab falls back here when no local `uvicorn` on port 8000) |
| Space (optional) | [HF Space — msme-training-runner](https://huggingface.co/spaces/di35117/msme-training-runner) — use **Files** tab for repo assets; **App** may show JupyterLab for maintenance |
| Training | **This notebook** (T4 / A100 GPU) |
| Model | `Qwen/Qwen2.5-1.5B-Instruct` (SFT warm start + GRPO; optional Unsloth if install succeeds) |
| Framework | Hugging Face `transformers` + `trl` GRPO (`train_grpo.py`) |

**Related:** [`inference.ipynb`](https://github.com/di35117/msme_env/blob/main/inference.ipynb) — trained vs random baseline demo after you have checkpoints.

Run all cells top to bottom.

## 1. Install dependencies

Clone the repo, install the package in editable mode, then run baselines + `train_grpo.py`. If Unsloth fails to install, training still works via `--no-unsloth` (HF Transformers path).

In [ ]:
import os
import shutil
import subprocess

# Repo path: Colab default under /content. On HF, set e.g. %env MSME_REPO=/data/msme_env
REPO = os.path.abspath(os.environ.get("MSME_REPO", "/content/msme_env"))
GIT_URL = os.environ.get("MSME_GIT", "https://github.com/di35117/msme_env.git")

def _have_repo(path: str) -> bool:
    return os.path.isfile(os.path.join(path, "train_grpo.py"))

if _have_repo(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "origin", "main"], check=False)
elif os.path.isdir(REPO):
    shutil.rmtree(REPO)
    subprocess.run(["git", "clone", GIT_URL, REPO], check=True)
else:
    parent = os.path.dirname(REPO)
    if parent:
        os.makedirs(parent, exist_ok=True)
    subprocess.run(["git", "clone", GIT_URL, REPO], check=True)

%cd {REPO}

!pip install -q -U pip
!pip install -q -e .
# Optional speedups (safe to skip if install errors):
# !pip install -q unsloth

print("Ready:", REPO)

## 2. Configuration

Add **`HF_TOKEN`** in Colab secrets (key icon) for higher Hub rate limits. Training uses the token for model downloads.

In [ ]:
import os

try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
except (ImportError, KeyError, ModuleNotFoundError):
    pass

if "HF_TOKEN" not in os.environ or not os.environ.get("HF_TOKEN"):
    print("WARNING: HF_TOKEN not set — Hub downloads may be slower or rate-limited.")

# --- Training hyperparameters (edit freely) ---
MODEL_ID = os.environ.get("MSME_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
NUM_EPISODES = int(os.environ.get("MSME_EPISODES", "30"))
MAX_STEPS = int(os.environ.get("MSME_MAX_STEPS", "90"))
SAVE_EVERY = int(os.environ.get("MSME_SAVE_EVERY", "2"))
OUTPUT_DIR = os.environ.get("MSME_OUTPUT_DIR", "msme_rl_run")
USE_UNSLOTH = os.environ.get("MSME_UNSLOTH", "1") not in ("0", "false", "False")

print(f"Model     : {MODEL_ID}")
print(f"Episodes  : {NUM_EPISODES}")
print(f"Steps/ep  : {MAX_STEPS}")
print(f"Save every: {SAVE_EVERY}")
print(f"Output    : {OUTPUT_DIR}")
print(f"Unsloth   : {USE_UNSLOTH}")

## 3. GPU check

In [ ]:
!nvidia-smi

## 4. Smoke test — environment

One `reset` proves `MSMERLEnvironment` imports and runs (same transition dynamics as the Space server).

In [ ]:
from server.msmeEnv_environment import MSMERLEnvironment

env = MSMERLEnvironment(max_episode_action_budget=MAX_STEPS)
obs = env.reset()
d = obs.__dict__ if hasattr(obs, "__dict__") else dict(obs)
print("Connected. Episode", d.get("episode"), "month", d.get("month"))
ps = d.get("portfolio_summary") or {}
print("NPA rate:", ps.get("npa_rate"), "| avg trust:", ps.get("avg_trust_score"))
print("Smoke test OK.")

## 5. Baseline + deterministic JSON

Required inputs for `scripts/generate_judge_artifacts.py` and `pre_submit_check.py`.

In [ ]:
!mkdir -p artifacts
!python scripts/run_baseline_eval.py --episodes 30 --output artifacts/baseline_rewards.json
!python scripts/run_deterministic_eval.py --seed 123 --episodes 5 --max_steps 90 --output artifacts/deterministic_eval.json
# Optional:
!python scripts/eval.py --episodes 5 --output artifacts/eval_report.json

## 6. GRPO training

Main entry point: [`train_grpo.py`](https://github.com/di35117/msme_env/blob/main/train_grpo.py). On Colab there is usually **no** env server on `localhost:8000`, so the script **falls back to in-process** `MSMERLEnvironment` automatically.

If Unsloth errors appear, re-run with `USE_UNSLOTH=False` in the config cell and add `--no-unsloth` below (or set env `MSME_UNSLOTH=0`).

In [ ]:
import subprocess
import sys

extra = [] if USE_UNSLOTH else ["--no-unsloth"]
cmd = [
    sys.executable, "train_grpo.py",
    "--episodes", str(NUM_EPISODES),
    "--max_steps_per_episode", str(MAX_STEPS),
    "--save_every", str(SAVE_EVERY),
    "--model", MODEL_ID,
    "--output_dir", OUTPUT_DIR,
] + extra
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 7. Judge artifacts, before/after, packaging check

Pick the **latest** `episode_*` checkpoint if you stopped early.

In [ ]:
import glob
import os
import subprocess
import sys

eps = sorted(glob.glob(os.path.join(OUTPUT_DIR, "episode_*")))
if not eps:
    raise RuntimeError(f"No checkpoints under {OUTPUT_DIR!r}. Train first.")
TRAINED = eps[-1]
print("Using checkpoint:", TRAINED)

subprocess.run([
    sys.executable, "scripts/generate_judge_artifacts.py",
    "--training_json", os.path.join(OUTPUT_DIR, "reward_curve.json"),
    "--baseline_json", "artifacts/baseline_rewards.json",
    "--output_dir", "artifacts",
], check=True)

subprocess.run([
    sys.executable, "scripts/inference_before_after.py",
    "--base", MODEL_ID,
    "--trained", TRAINED,
    "--output-dir", OUTPUT_DIR,
    "--episodes", "5",
    "--max-steps", str(MAX_STEPS),
], check=True)

subprocess.run([sys.executable, "scripts/make_hero.py", "--run-dir", OUTPUT_DIR], check=True)
subprocess.run([sys.executable, "scripts/pre_submit_check.py"], check=True)
print("Done.")

## 8. Reward curve (inline)

Plots are also under `artifacts/` and `OUTPUT_DIR/` after training.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

p = Path(OUTPUT_DIR) / "reward_curve.png"
if p.is_file():
    display(Image(filename=str(p)))
else:
    print("No reward_curve.png yet — check", OUTPUT_DIR)

## 9. Optional — copy outputs to Google Drive

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/msme_env_outputs")
dest.mkdir(parents=True, exist_ok=True)
for name in ["artifacts", OUTPUT_DIR]:
    src = Path(name)
    if src.exists():
        shutil.copytree(src, dest / src.name, dirs_exist_ok=True)
print("Copied to", dest)